# Maritime Distance Calculator using Official Searoute-py
Calculate port-to-port maritime distances using the official searoute library API to compare with Signal Ocean data

In [1]:
import re
import pandas as pd
import numpy as np
import json
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import warnings

warnings.filterwarnings("ignore")

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [2]:
def parse_sql_ports(sql_file_path: str) -> pd.DataFrame:
    """Parse SQL INSERT statements and extract port data (lon, lat, name, country)"""
    ports_data = []

    with open(sql_file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Find all INSERT statements
    pattern = r"INSERT INTO `ports`.*?VALUES\s*(.*?);"
    matches = re.finditer(pattern, content, re.DOTALL)

    for match in matches:
        values_str = match.group(1)
        # Split by rows
        rows = re.findall(r"\(\d+,\s*'[^']*',\s*'[^']*'.*?\)", values_str)

        for row in rows:
            # Extract values: (id, name, country, lat, lon, region)
            parts = re.match(
                r"\((\d+),\s*'([^']*)',\s*'([^']*)',\s*([\d.-]+),\s*([\d.-]+),\s*(NULL|'[^']*')\)",
                row,
            )
            if parts:
                port_id = int(parts.group(1))
                name = parts.group(2)
                country = parts.group(3)
                lat = float(parts.group(4))
                lon = float(parts.group(5))

                ports_data.append(
                    {
                        "id": port_id,
                        "name": name,
                        "country": country,
                        "lat": lat,
                        "lon": lon,
                        "lon_lat": (lon, lat),  # Tuple format for searoute
                    }
                )

    return pd.DataFrame(ports_data)


# Load ports
sql_path = (
    "/home/faisalhrbk/Projects/shortest-route/ports _1000.sql"  # Corrected filename
)
ports_df = parse_sql_ports(sql_path)

print(f"✓ Loaded {len(ports_df)} ports from SQL file")
print(f"\nFirst 5 ports:")
print(ports_df[["id", "name", "country", "lat", "lon"]].head())
print(f"\nPort database statistics:")
print(f"  Countries: {ports_df['country'].nunique()}")
print(f"  Lat range: {ports_df['lat'].min():.2f}° to {ports_df['lat'].max():.2f}°")
print(f"  Lon range: {ports_df['lon'].min():.2f}° to {ports_df['lon'].max():.2f}°")

✓ Loaded 995 ports from SQL file

First 5 ports:
   id      name  country        lat       lon
0   1  A Coruna    Spain  43.362243 -8.384353
1   2  Aabenraa  Denmark  55.033330  9.433330
2   3    Aaheim   Norway  62.050000  5.516670
3   4     Aakra   Norway  59.783330  6.100000
4   5   Aalborg  Denmark  57.050000  9.916670

Port database statistics:
  Countries: 123
  Lat range: -65.50° to 78.58°
  Lon range: -179.47° to 179.37°


In [3]:
# OFFICIAL SEAROUTE-PY IMPLEMENTATION
# Based on: https://github.com/genthalili/searoute-py
# Reference: https://github.com/genthalili/searoute-py#usage

import searoute as sr


def calculate_maritime_route(
    origin: Tuple[float, float],
    destination: Tuple[float, float],
    units: str = "naut",
    include_ports: bool = False,
    restrictions: List[str] = None,
) -> Dict:
    """
    Calculate maritime distance using official searoute-py library

    Parameters:
    -----------
    origin : tuple(lon, lat)
        Starting point coordinates
    destination : tuple(lon, lat)
        Ending point coordinates
    units : str
        Distance unit. Default 'naut' (nautical miles).
        Options: 'km', 'mi', 'naut', 'm', 'ft', 'in', 'deg', 'cen', 'rad', 'yd'
    include_ports : bool
        Include nearest port information in results
    restrictions : list
        Maritime passages to avoid
        Options: 'babalmandab', 'bosporus', 'gibraltar', 'suez', 'panama',
                 'ormuz', 'northwest', 'malacca', 'sunda', 'chili', 'south_africa'

    Returns:
    --------
    dict with keys: distance, units, duration_hours, route_geojson, status
    """

    try:
        # Build parameters
        route_params = {
            "origin": origin,
            "destination": destination,
            "units": units,
            "append_orig_dest": True,  # Include origin/destination in coordinates
        }

        if restrictions:
            route_params["restrictions"] = restrictions

        if include_ports:
            route_params["include_ports"] = True

        # Calculate route (official API call)
        route = sr.searoute(**route_params)

        # Extract properties (returned as GeoJSON Feature)
        props = route.properties

        result = {
            "distance": props.get("length"),
            "units": props.get("units"),
            "duration_hours": props.get("duration_hours"),
            "status": "success",
            "route_geojson": route,
        }

        # Add port info if requested
        if include_ports:
            result["port_origin"] = props.get("port_origin")
            result["port_dest"] = props.get("port_dest")

        return result

    except Exception as e:
        return {"distance": None, "status": f"error: {str(e)}", "route_geojson": None}


# Test with sample routes
print("\n" + "=" * 70)
print("TESTING OFFICIAL SEAROUTE-PY API")
print("=" * 70)

# Test 1: Simple route without ports
print("\n1. Basic route (A Coruna, Spain → Singapore):")
origin1 = (ports_df.iloc[0]["lon"], ports_df.iloc[0]["lat"])  # A Coruna
dest1 = (103.8, 1.3)  # Singapore

route1 = calculate_maritime_route(origin1, dest1, units="naut")
print(f"   Origin: {ports_df.iloc[0]['name']}, {origin1}")
print(f"   Distance: {route1['distance']:.2f} {route1['units']}")
print(f"   Status: {route1['status']}")

# Test 2: Route with ports
print("\n2. Route with ports information:")
origin2 = (ports_df.iloc[5]["lon"], ports_df.iloc[5]["lat"])  # Port 6
dest2 = (ports_df.iloc[20]["lon"], ports_df.iloc[20]["lat"])  # Port 21

route2 = calculate_maritime_route(origin2, dest2, units="naut", include_ports=True)
print(f"   Distance: {route2['distance']:.2f} {route2['units']}")
if "port_origin" in route2:
    print(f"   Nearest Origin Port: {route2['port_origin']}")
print(f"   Status: {route2['status']}")


TESTING OFFICIAL SEAROUTE-PY API

1. Basic route (A Coruna, Spain → Singapore):
   Origin: A Coruna, (np.float64(-8.384353), np.float64(43.362243))
   Distance: 7630.35 naut
   Status: success

2. Route with ports information:
   Distance: 649.00 naut
   Nearest Origin Port: {'cty': 'Norway', 'name': 'Kristiansand S.Chris', 'port': 'NOKRS', 't': 1.0, 'x': 7.99736, 'y': 58.145721, 'share': 1}
   Status: success


In [4]:
import time

total_ports = len(ports_df)
num_samples = total_ports * (total_ports - 1) // 2

print("\n" + "=" * 70)
print(f"CALCULATING DISTANCES FOR ALL PORT PAIRS ({num_samples:,} total)")
print("=" * 70)

results = []
success_count = 0
failed_count = 0
pair_number = 0

for idx1 in range(total_ports):
    port1 = ports_df.iloc[idx1]

    for idx2 in range(idx1 + 1, total_ports):
        port2 = ports_df.iloc[idx2]
        pair_number += 1

        # Calculate route using official searoute API
        origin = (port1["lon"], port1["lat"])
        destination = (port2["lon"], port2["lat"])

        route_result = calculate_maritime_route(
            origin=origin,
            destination=destination,
            units="naut",  # Nautical miles
            include_ports=False,
        )

        # Record result
        result = {
            "pair_number": pair_number,
            "origin_port": port1["name"],
            "origin_country": port1["country"],
            "origin_lat": port1["lat"],
            "origin_lon": port1["lon"],
            "destination_port": port2["name"],
            "destination_country": port2["country"],
            "dest_lat": port2["lat"],
            "dest_lon": port2["lon"],
            "distance_nm": route_result["distance"],
            "distance_km": route_result["distance"] * 1.852
            if route_result["distance"]
            else None,
            "duration_hours": route_result.get("duration_hours"),
            "status": route_result["status"],
        }

        results.append(result)

        if route_result["distance"]:
            success_count += 1
        else:
            failed_count += 1

        if pair_number % 100 == 0:
            print(f"  ✓ Processed {pair_number:,}/{num_samples:,} routes...")

        time.sleep(0.01)  # Small delay to avoid overload

print(f"\n{'=' * 70}")
print("CALCULATION COMPLETE")
print(f"{'=' * 70}")
print(f"✓ Successful: {success_count:,}/{num_samples:,}")
print(f"✗ Failed: {failed_count:,}/{num_samples:,}")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n📊 DISTANCE STATISTICS:")
print(f"  Mean distance: {results_df['distance_nm'].mean():.2f} NM")
print(f"  Median distance: {results_df['distance_nm'].median():.2f} NM")
print(f"  Min distance: {results_df['distance_nm'].min():.2f} NM")
print(f"  Max distance: {results_df['distance_nm'].max():.2f} NM")
print(f"  Std dev: {results_df['distance_nm'].std():.2f} NM")

print("\n📋 SAMPLE RESULTS:")
print(
    results_df[
        ["pair_number", "origin_port", "destination_port", "distance_nm", "status"]
    ]
    .head(10)
    .to_string(index=False)
)


CALCULATING DISTANCES FOR ALL PORT PAIRS (494,515 total)
  ✓ Processed 100/494,515 routes...
  ✓ Processed 200/494,515 routes...
  ✓ Processed 300/494,515 routes...
  ✓ Processed 400/494,515 routes...
  ✓ Processed 500/494,515 routes...
  ✓ Processed 600/494,515 routes...
  ✓ Processed 700/494,515 routes...
  ✓ Processed 800/494,515 routes...
  ✓ Processed 900/494,515 routes...
  ✓ Processed 1,000/494,515 routes...
  ✓ Processed 1,100/494,515 routes...
  ✓ Processed 1,200/494,515 routes...
  ✓ Processed 1,300/494,515 routes...
  ✓ Processed 1,400/494,515 routes...
  ✓ Processed 1,500/494,515 routes...
  ✓ Processed 1,600/494,515 routes...
  ✓ Processed 1,700/494,515 routes...
  ✓ Processed 1,800/494,515 routes...
  ✓ Processed 1,900/494,515 routes...
  ✓ Processed 2,000/494,515 routes...
  ✓ Processed 2,100/494,515 routes...
  ✓ Processed 2,200/494,515 routes...
  ✓ Processed 2,300/494,515 routes...
  ✓ Processed 2,400/494,515 routes...
  ✓ Processed 2,500/494,515 routes...
  ✓ Proces

KeyboardInterrupt: 

In [ ]:
# Export results for validation
print("\n" + "=" * 70)
print("EXPORTING RESULTS")
print("=" * 70)

# 1. CSV export
csv_path = "/home/faisalhrbk/Projects/shortest-route/searoute_distances.csv"
results_df.to_csv(csv_path, index=False)
print(f"✓ CSV Export: {csv_path}")

# 2. JSON export
json_path = "/home/faisalhrbk/Projects/shortest-route/searoute_distances.json"
results_df.to_json(json_path, orient="records", indent=2)
print(f"✓ JSON Export: {json_path}")

# 3. GeoJSON export for visualization
geojson_features = []
for _, row in results_df.iterrows():
    feature = {
        "type": "Feature",
        "properties": {
            "pair_number": row["pair_number"],
            "origin": row["origin_port"],
            "destination": row["destination_port"],
            "distance_nm": row["distance_nm"],
            "distance_km": row["distance_km"],
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [row["origin_lon"], row["origin_lat"]],
                [row["dest_lon"], row["dest_lat"]],
            ],
        },
    }
    geojson_features.append(feature)

geojson_data = {"type": "FeatureCollection", "features": geojson_features}

geojson_path = "/home/faisalhrbk/Projects/shortest-route/routes.geojson"
with open(geojson_path, "w") as f:
    json.dump(geojson_data, f, indent=2)
print(f"✓ GeoJSON Export: {geojson_path} (for visualization)")

print(f"\n{'=' * 70}")
print("READY FOR VALIDATION AGAINST SIGNAL OCEAN")
print(f"{'=' * 70}")

In [ ]:
def compare_with_signal_ocean(
    our_results: pd.DataFrame, signal_ocean_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Compare calculated distances with Signal Ocean reference data

    Parameters:
    -----------
    our_results : DataFrame
        Our calculated distances (searoute_distances.csv)
    signal_ocean_data : DataFrame
        Signal Ocean reference data with 'signal_ocean_distance' column

    Returns:
    --------
    comparison_df : DataFrame with metrics
    """

    comparison = our_results.copy()

    # Merge Signal Ocean data
    if len(signal_ocean_data) == len(our_results):
        comparison["signal_ocean_distance"] = signal_ocean_data[
            "signal_ocean_distance"
        ].values
    else:
        print(
            f"⚠ Warning: Data size mismatch - {len(our_results)} vs {len(signal_ocean_data)}"
        )
        return None

    # Calculate metrics
    comparison["difference_nm"] = abs(
        comparison["distance_nm"] - comparison["signal_ocean_distance"]
    )
    comparison["difference_km"] = comparison["difference_nm"] * 1.852
    comparison["percent_diff"] = (
        comparison["difference_nm"] / comparison["signal_ocean_distance"] * 100
    ).round(2)

    # Calculate statistics
    mean_diff = comparison["difference_nm"].mean()
    rmse = (comparison["difference_nm"] ** 2).mean() ** 0.5
    mae = comparison["difference_nm"].mean()  # Mean Absolute Error
    mean_percent = comparison["percent_diff"].mean()
    max_percent = comparison["percent_diff"].max()

    print("\n" + "=" * 70)
    print("VALIDATION REPORT: Searoute vs Signal Ocean")
    print("=" * 70)

    print(f"\n📊 ACCURACY METRICS:")
    print(f"  Mean Absolute Error: {mae:.2f} NM")
    print(f"  RMSE (Root Mean Square Error): {rmse:.2f} NM")
    print(f"  Mean Percent Difference: {mean_percent:.3f}%")
    print(f"  Max Percent Difference: {max_percent:.3f}%")

    print(f"\n✅ ACCURACY LEVEL:")
    if mean_percent < 2:
        print(f"  EXCELLENT - Within 2% ✓✓")
    elif mean_percent < 5:
        print(f"  GOOD - Within 5% ✓")
    elif mean_percent < 10:
        print(f"  ACCEPTABLE - Within 10% ◐")
    else:
        print(f"  NEEDS IMPROVEMENT - Above 10% ✗")

    print(f"\n📈 ERROR DISTRIBUTION:")
    print(f"  Within 1%: {len(comparison[comparison['percent_diff'] <= 1])} routes")
    print(f"  Within 2%: {len(comparison[comparison['percent_diff'] <= 2])} routes")
    print(f"  Within 5%: {len(comparison[comparison['percent_diff'] <= 5])} routes")
    print(f"  Within 10%: {len(comparison[comparison['percent_diff'] <= 10])} routes")

    print(f"\n📋 SAMPLE COMPARISONS (Top 10):")
    sample_cols = [
        "pair_number",
        "origin_port",
        "destination_port",
        "distance_nm",
        "signal_ocean_distance",
        "difference_nm",
        "percent_diff",
    ]
    print(comparison[sample_cols].head(10).to_string(index=False))

    return comparison


print("✓ Comparison function ready")
print("\nUsage:")
print("  signal_ocean_df = pd.read_csv('signal_ocean_data.csv')")
print("  comparison = compare_with_signal_ocean(results_df, signal_ocean_df)")